# Wave 2 — LLMHandler / Retriever / DocumentParser / DoclingParser

**먼저 `00_setup.ipynb`를 실행하세요.**

## 1. LLMProvider 테스트

In [ ]:
%%ipytest
import pytest
from unittest.mock import patch, MagicMock
from app.core.llm_handler import LLMProvider
from app.factories.config import Config, LLMConfig, EmbeddingConfig, VectorDBConfig, PromptConfig

@pytest.fixture
def ollama_cfg():
    return Config(id='T', llm=LLMConfig(provider='ollama', model_name='test-model', base_url='http://localhost:11434'),
                  embedding=EmbeddingConfig(), vector_db=VectorDBConfig(), prompt=PromptConfig())

@pytest.fixture
def openai_cfg():
    return Config(id='T', llm=LLMConfig(provider='openai', model_name='gpt-4o'),
                  embedding=EmbeddingConfig(), vector_db=VectorDBConfig(), prompt=PromptConfig())

@pytest.fixture
def anthropic_cfg():
    return Config(id='T', llm=LLMConfig(provider='anthropic', model_name='claude-sonnet-4-6'),
                  embedding=EmbeddingConfig(), vector_db=VectorDBConfig(), prompt=PromptConfig())

def test_openai_calls_chatopenai(openai_cfg):
    with patch('app.core.llm_handler.ChatOpenAI') as Mock:
        LLMProvider.get_model(openai_cfg)
        assert Mock.called
        assert Mock.call_args.kwargs['model'] == 'gpt-4o'
        assert Mock.call_args.kwargs['streaming'] is True

def test_ollama_calls_chatollama(ollama_cfg):
    with patch('app.core.llm_handler.ChatOllama') as Mock:
        LLMProvider.get_model(ollama_cfg)
        assert Mock.called
        assert Mock.call_args.kwargs['model'] == 'test-model'
        assert Mock.call_args.kwargs['num_ctx'] == 8192

def test_anthropic_calls_chatanthropic(anthropic_cfg):
    with patch('app.core.llm_handler.ChatAnthropic') as Mock:
        LLMProvider.get_model(anthropic_cfg)
        assert Mock.called

def test_unknown_provider_raises_valueerror(ollama_cfg):
    ollama_cfg.llm.provider = 'unknown'
    with pytest.raises(ValueError) as exc_info:
        LLMProvider.get_model(ollama_cfg)
    assert 'unknown' in str(exc_info.value)
    assert 'openai' in str(exc_info.value)

## 2. KnowledgeRetriever 테스트

In [ ]:
%%ipytest
import pytest
from unittest.mock import patch, MagicMock
from langchain_core.documents import Document
from app.factories.config import Config, LLMConfig, EmbeddingConfig, VectorDBConfig, PromptConfig

def make_doc(content, source, score, image_path=None):
    meta = {'source': source}
    if image_path:
        meta['image_path'] = image_path
    return (Document(page_content=content, metadata=meta), score)

@pytest.fixture
def retriever():
    cfg = Config(id='T', llm=LLMConfig(provider='ollama', model_name='m', base_url='http://localhost:11434'),
                 embedding=EmbeddingConfig(), vector_db=VectorDBConfig(retrieval_k=3, score_threshold=0.7), prompt=PromptConfig())
    with patch('app.core.retriever.OllamaEmbeddings'), patch('app.core.retriever.Chroma') as MockChroma:
        mock_db = MagicMock()
        MockChroma.return_value = mock_db
        from app.core.retriever import KnowledgeRetriever
        r = KnowledgeRetriever(cfg)
        r._mock_db = mock_db
        yield r

def test_base_mode_returns_empty(retriever):
    ctx, imgs, chunk_map = retriever.get_context('q', 'base')
    assert ctx == '' and imgs == [] and chunk_map == {}

def test_base_mode_skips_similarity_search(retriever):
    retriever.get_context('q', 'base')
    retriever._mock_db.similarity_search_with_score.assert_not_called()

def test_rag_filters_below_threshold(retriever):
    retriever._mock_db.similarity_search_with_score.return_value = [make_doc('내용', 'test.pdf', 0.5)]
    ctx, _, chunk_map = retriever.get_context('q', 'rag')
    assert ctx == '' and chunk_map == {}

def test_rag_includes_above_threshold(retriever):
    retriever._mock_db.similarity_search_with_score.return_value = [make_doc('중요 내용', 'test.pdf', 0.85)]
    ctx, _, chunk_map = retriever.get_context('q', 'rag')
    assert '중요 내용' in ctx
    assert 'test.pdf' in chunk_map

def test_rag_collects_image_paths(retriever):
    retriever._mock_db.similarity_search_with_score.return_value = [
        make_doc('내용', 'test.pdf', 0.9, image_path='/img/1.png')]
    _, imgs, _ = retriever.get_context('q', 'rag')
    assert '/img/1.png' in imgs

def test_graph_mode_appends_graph_text(retriever):
    retriever._mock_db.similarity_search_with_score.return_value = [make_doc('내용', 'test.pdf', 0.9)]
    ctx, _, _ = retriever.get_context('q', 'graph')
    assert '[관계 정보]' in ctx

## 3. DocumentParser 테스트

In [ ]:
%%ipytest
import re
from langchain_core.documents import Document
from pipeline.data_loader import DocumentParser

def test_assigns_chunk_id():
    parser = DocumentParser()
    doc = Document(page_content='테스트 내용', metadata={})
    result = parser.split([doc])
    assert 'chunk_id' in result[0].metadata

def test_chunk_id_is_12_char_hex():
    parser = DocumentParser()
    doc = Document(page_content='테스트 내용', metadata={})
    result = parser.split([doc])
    assert re.fullmatch(r'[0-9a-f]{12}', result[0].metadata['chunk_id'])

def test_chunk_id_is_deterministic():
    parser = DocumentParser()
    doc = Document(page_content='결정적 내용', metadata={})
    id1 = parser.split([doc])[0].metadata['chunk_id']
    id2 = parser.split([doc])[0].metadata['chunk_id']
    assert id1 == id2

def test_empty_input_returns_empty():
    parser = DocumentParser()
    assert parser.split([]) == []

def test_large_doc_splits_into_multiple_chunks():
    parser = DocumentParser(chunk_size=50, chunk_overlap=5)
    doc = Document(page_content='가나다라마바사아자차 ' * 30, metadata={})
    assert len(parser.split([doc])) > 1

## 4. DoclingParser 테스트

In [ ]:
%%ipytest
from unittest.mock import patch, MagicMock
from langchain_core.documents import Document
from pipeline.parsers.docling_parser import DoclingParser

def test_no_headings_returns_full_text():
    result = DoclingParser._split_by_heading('헤딩 없는 텍스트')
    assert result == ['헤딩 없는 텍스트']

def test_single_heading_splits_two_parts():
    result = DoclingParser._split_by_heading('도입\n## 섹션 1\n본문')
    assert len(result) == 2

def test_multiple_headings_splits_all():
    text = '도입\n## A\n내용A\n## B\n내용B\n## C\n내용C'
    result = DoclingParser._split_by_heading(text)
    assert len(result) == 4

def test_ignores_h1_and_h3():
    text = '# H1\n내용\n### H3\n내용'
    result = DoclingParser._split_by_heading(text)
    assert result == [text]

def test_parse_sets_metadata():
    with patch('pipeline.parsers.docling_parser.DocumentConverter') as Mock:
        mock_result = MagicMock()
        mock_result.document.export_to_markdown.return_value = '도입\n## 섹션 1\n내용'
        Mock.return_value.convert.return_value = mock_result
        parser = DoclingParser()
        docs = parser.parse('/data/manual.pdf')
    for doc in docs:
        assert doc.metadata['doc_type'] == 'manual'
        assert doc.metadata['parser'] == 'docling'
        assert doc.metadata['source'] == 'manual.pdf'